#### Adds:
* SQL
* Database Concepts
* Query Logging
* Analytics
* User Activity Tracking
* Production Monitoring

### Imports

In [1]:
import sqlite3
import pandas as pd
from datetime import datetime

### Create Database

In [2]:
connection = sqlite3.connect("data/query_logs.db")

cursor = connection.cursor()

print("Database created successfully!")

Database created successfully!


### Create Table

In [3]:
cursor.execute("""

CREATE TABLE IF NOT EXISTS query_logs (

    id INTEGER PRIMARY KEY AUTOINCREMENT,

    timestamp TEXT,

    question TEXT,

    answer TEXT,

    source TEXT

)

""")

connection.commit()

print("Table created successfully!")

Table created successfully!


### Insert Function

In [4]:
def log_query(question, answer, source):

    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    cursor.execute("""

    INSERT INTO query_logs
    (timestamp, question, answer, source)

    VALUES (?, ?, ?, ?)

    """,

    (timestamp, question, answer, source))

    connection.commit()

### Test Logging

In [5]:
log_query(

    question="How many leave days are allowed?",

    answer="Employees receive 20 annual leave days.",

    source="employee_handbook.txt"

)

print("Query logged!")

Query logged!


### Add Multiple Records

In [6]:
log_query(

    "Who works in ML Engineering?",

    "Bob works in ML Engineering.",

    "employees.csv"

)

log_query(

    "What monitoring tools are used?",

    "Prometheus and Grafana",

    "cloud_migration_guide.txt"

)

print("Sample data inserted!")

Sample data inserted!


### Read Database

In [7]:
df = pd.read_sql(

    "SELECT * FROM query_logs",

    connection

)

df

,id,timestamp,question,answer,source
0,1,2026-06-23 10:31:04,How many leave days are allowed?,Employees receive 20 annual leave days.,employee_handbook.txt
1,2,2026-06-23 10:31:27,Who works in ML Engineering?,Bob works in ML Engineering.,employees.csv
2,3,2026-06-23 10:31:27,What monitoring tools are used?,Prometheus and Grafana,cloud_migration_guide.txt


###  Total Queries

In [8]:
cursor.execute(

    "SELECT COUNT(*) FROM query_logs"

)

total_queries = cursor.fetchone()[0]

print("Total Queries:", total_queries)

Total Queries: 3


### Most Used Sources

In [9]:
query = """

SELECT source,
COUNT(*) AS total

FROM query_logs

GROUP BY source

ORDER BY total DESC

"""

pd.read_sql(query, connection)

,source,total
0,employees.csv,1
1,employee_handbook.txt,1
2,cloud_migration_guide.txt,1


### Questions History

In [10]:
query = """

SELECT timestamp,
question

FROM query_logs

ORDER BY timestamp DESC

"""

pd.read_sql(query, connection)

,timestamp,question
0,2026-06-23 10:31:27,Who works in ML Engineering?
1,2026-06-23 10:31:27,What monitoring tools are used?
2,2026-06-23 10:31:04,How many leave days are allowed?


### Questions Per Source

In [11]:
query = """

SELECT source,
COUNT(*) AS frequency

FROM query_logs

GROUP BY source

"""

pd.read_sql(query, connection)

,source,frequency
0,cloud_migration_guide.txt,1
1,employee_handbook.txt,1
2,employees.csv,1


### Top 5 Recent Queries


In [12]:
query = """

SELECT *

FROM query_logs

ORDER BY id DESC

LIMIT 5

"""

pd.read_sql(query, connection)

,id,timestamp,question,answer,source
0,3,2026-06-23 10:31:27,What monitoring tools are used?,Prometheus and Grafana,cloud_migration_guide.txt
1,2,2026-06-23 10:31:27,Who works in ML Engineering?,Bob works in ML Engineering.,employees.csv
2,1,2026-06-23 10:31:04,How many leave days are allowed?,Employees receive 20 annual leave days.,employee_handbook.txt


###  Export Analytics

In [13]:
analytics_df = pd.read_sql(

    "SELECT * FROM query_logs",

    connection

)

analytics_df.to_csv(

    "outputs/query_analytics.csv",

    index=False

)

print("Analytics exported successfully!")

Analytics exported successfully!


### Verify Output Files

In [14]:
import os

print(os.listdir("outputs"))

['query_analytics.csv']


### Close Database

In [15]:
connection.close()

print("Database connection closed.")

Database connection closed.
